# Using Skills to Extend the Agent Capabilities
In this notebook, we will learn how to extend the agent’s context by providing additional information—such as business logic, data mappings, or domain-specific rules—that cannot be reliably inferred from the data alone.


<figure>
 <img src="../assets/chapter_2.png" width="60%" align="center"/></a>
<figcaption> Prompt Template Architecture </figcaption>
</figure>

<br>
<br />

## Setting the Database Connection

The below code enables us to connect to Postgres (or DuckDB) using the `get_ibis_connection` function:

In [ ]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection



Setting a connection to the Postgres database:

In [ ]:
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

Or, setting a connection to the DuckDB database:

In [ ]:
# csv_path = project_root + "/data/air_traffic_gold.csv"
# con = get_ibis_connection(
#     backend="duckdb",
#     duckdb_csv_path=csv_path,
# )

LLM settings:

In [ ]:
base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

## Loading and Injecting Skills

In [ ]:
from sql_ai_agent.skill_manager import SkillManager

In [ ]:
# Initialize skill manager
skill_manager = SkillManager(skills_dir= "../skills")

# List available skills
print("Available skills:")
for skill in skill_manager.list_skills():
    print(f"  - {skill}")



In [ ]:
# Load the SFO air traffic skill
sfo_skill = skill_manager.load_skill("sfo_air_traffic_context")
print(f"\n✓ Loaded skill: {len(sfo_skill):,} characters")


In [ ]:
print(sfo_skill)

In [ ]:

from sql_ai_agent.SqlAgent import SqlAgent

agent_with_skill = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    tbl_name=tbl_name,
    fallback=False,
    fallback_model=model,
    skill=True,
    skills_dir="../skills",
)

agent_without_skill = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    tbl_name=tbl_name,
    fallback=False,
    fallback_model=model,
    skill=False,
)

print("✓ Agent initialized")


In [ ]:
complex_question = """
  Which international airlines showed the biggest growth from 2023 to 2024?
  Show the top 10 airlines ranked by percentage increase in passenger volume.
  For each airline, show their 2023 total, 2024 total, and calculate year-over-year growth

  Make sure to exclude transit passengers and avoid double-counting from code share agreements.
  """

In [ ]:
result_without_skill = agent_without_skill.ask_question(
    question= complex_question,
    distinct_char_values = True,
    verbose=True,
    trials= 0
)

In [ ]:
result_with_skill = agent_with_skill.ask_question(
    question=complex_question,
    distinct_char_values = True,
    verbose=True,
    trials= 0
)
